# 06_collect — 팀 결과 통합 점검과 보고서 표·그림 (담당: core)

`outputs/{logit, xgb, xgb_mono}/`에 올라온 결과를 읽어 두 가지를 합니다.

1. **점검:** 세 모형이 같은 버전(config·lib_prep 해시), 같은 열(contract), 같은 B, 같은 출력 형식으로 돌았는지 확인합니다. 하나라도 다르면 표를 만들기 전에 멈추고 무엇이 다른지 알려 줍니다.
2. **보고서용:** 표 4(`report/tables/t4_validation_summary.csv`), 그림 2·3·4(`report/figures/fig2~4`)를 만듭니다.

아직 올라오지 않은 모형은 "없음"으로 표시하고 있는 모형만으로 만듭니다. 그림 4(ΔSR)는 ②가 있어야 그립니다.
B=100 본 실행 결과가 모두 모이면 이 노트북만 다시 실행하면 됩니다.

In [1]:
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
%run -i ./lib_eval.ipynb
import json, joblib
import matplotlib.pyplot as plt
from matplotlib import font_manager
for name in ("AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
    if name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = name; break
plt.rcParams["axes.unicode_minus"] = False
MODELS = {"logit": "① 로지스틱", "xgb": "② XGBoost", "xgb_mono": "③ 단조 XGBoost"}
TAB_DIR = REPORT_DIR / "tables"; FIG_DIR = REPORT_DIR / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
NOW = dict(config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH)
print("지금 버전:", NOW)

ModuleNotFoundError: No module named 'nbformat'

In [ ]:
# 1. 점검 (모형마다 PASS/FAIL, 없는 모형은 건너뜀)
REQ_RES = ["b", "a_star", "tau_star", "sr_max", "auc", "brier"]
REQ_CURVE = ["b", "rate", "sr", "tau"]
REQ_SUMMARY = ["model", "a_star", "tau_star_median", "columns", "config_hash", "lib_prep_hash"]
cols_contract = {"logit": json.load(open(CONTRACT_DIR / "columns_logit.json")),
                 "xgb": json.load(open(CONTRACT_DIR / "columns_xgb.json"))}
cols_contract["xgb_mono"] = cols_contract["xgb"]

have, report = {}, []
for m, label in MODELS.items():
    d = OUT_DIR / m
    if not (d / "summary.json").exists():
        report.append((label, "없음", "outputs/%s/summary.json 없음" % m)); continue
    s = json.load(open(d / "summary.json"))
    res = pd.read_csv(d / "bootstrap_results.csv"); cur = pd.read_csv(d / "bootstrap_curves.csv")
    n_models = len(list((d / "models_b").glob("model_b*.joblib"))) if (d / "models_b").exists() else 0
    checks = {
        "summary 공통 키": all(k in s for k in REQ_SUMMARY),
        "config 해시 = 지금": s.get("config_hash") == NOW["config_hash"],
        "lib_prep 해시 = 지금": s.get("lib_prep_hash") == NOW["lib_prep_hash"],
        "열 = contract": s.get("columns") == cols_contract[m],
        "결과 열 형식": list(res.columns) == REQ_RES,
        "곡선 열 형식": list(cur.columns) == REQ_CURVE,
        "회차 수 = B": len(res) == B,
        "보관 모형 수 = 회차 수": n_models == len(res),
        "최종 모형 있음": (d / "final_model.joblib").exists(),
        "a* = 회차 중앙값": np.isclose(s.get("a_star", np.nan), res["a_star"].median()),
    }
    bad = [k for k, v in checks.items() if not v]
    report.append((label, "PASS" if not bad else "FAIL", ", ".join(bad) if bad else f"B={len(res)}"))
    have[m] = dict(summary=s, res=res, curves=cur)
chk = pd.DataFrame(report, columns=["모형", "판정", "비고"])
print(chk.to_string(index=False))
FAILED = chk["판정"].eq("FAIL").any()
if FAILED:
    print("\nFAIL이 있으면 표와 그림은 '시험용'으로만 만듭니다. 담당자에게 비고의 항목을 고쳐 달라고 요청하세요.")

### 1-1. 분할 지문 비교 (4단계: 세 모형이 같은 분할)
각 모형 `summary.json`의 `split_hash`(B회 OOB 인덱스를 이어 붙인 sha256 앞 12자)가 모두 같아야 회차끼리 짝지어 뺄 수 있습니다. 다르거나 없으면 FAIL로 표시하고 표·그림을 `_시험용`으로 만듭니다.

In [ ]:
# 1-1. 분할 지문 비교
split = {m: have[m]["summary"].get("split_hash") for m in have}
n_rows = {m: have[m]["summary"].get("n_samples") for m in have}
for m, h in split.items():
    print(f"{MODELS[m]:<12} split_hash={h}  n_samples={n_rows[m]}")
missing = [MODELS[m] for m, h in split.items() if not h]
SPLIT_OK = bool(split) and not missing and len(set(split.values())) == 1
if missing:
    print("split_hash 없음 (split_hash 추가 전 실행):", ", ".join(missing), "→ 새 model_template·02_logit으로 다시 실행 필요")
elif len(set(split.values())) > 1:
    print("분할 지문이 서로 다름 → 같은 n·B·seed로 돌았는지 확인. 짝지은 비교를 믿을 수 없음")
else:
    print("분할 지문 일치:", next(iter(split.values())) if split else "-")
if not SPLIT_OK:
    FAILED = True

In [ ]:
# 2. 표 4: Dev OOB 요약 (회차 B개)
rows = []
for m, label in MODELS.items():
    if m not in have: continue
    r = have[m]["res"]
    rows.append({"모형": label, "B": len(r), "a* 중앙값": r["a_star"].median(),
                 "a* 범위": f"{r['a_star'].min():.2f}~{r['a_star'].max():.2f}",
                 "SR* 평균": r["sr_max"].mean(), "AUC 평균": r["auc"].mean(), "Brier 평균": r["brier"].mean()})
t4 = pd.DataFrame(rows).round(4)
tag = "_시험용" if FAILED else ""
t4.to_csv(TAB_DIR / f"t4_validation_summary{tag}.csv", index=False, encoding="utf-8-sig")
print(t4.to_string(index=False))

### 2-1. 승인율 a* = 평균 곡선의 최댓값 (5단계)
같은 승인율의 SR B개를 평균한 곡선에서 `MIN_RATE` 이상 구간의 최댓값 지점을 a*로 씁니다(기대 SR 최대화). 최댓값에서 0.005 이내 승인율 구간을 함께 적고, 회차별 최적값의 중앙값은 견고성 기록으로 둡니다.
결과: `report/tables/t4b_astar_mean_curve.csv`

In [ ]:
# 2-1. 평균 곡선 a*
NEAR_MAX = 0.005   # 아키텍처 5단계: 최댓값에서 0.005 이내 구간도 보고
A_STAR, rows = {}, []
for m, label in MODELS.items():
    if m not in have: continue
    cur = have[m]["curves"].assign(rate=lambda d: d["rate"].round(4))
    mean_c = cur[cur["rate"] >= MIN_RATE - 1e-9].groupby("rate")["sr"].mean().dropna()
    a_m, sr_m = float(mean_c.idxmax()), float(mean_c.max())
    near = mean_c[mean_c >= sr_m - NEAR_MAX].index
    A_STAR[m] = a_m
    rows.append({"모형": label, "a* (평균 곡선 최대)": a_m, "평균 곡선 SR 최대": sr_m,
                 "0.005 이내 구간": f"{near.min():.2f}~{near.max():.2f}",
                 "회차별 a*_b 중앙값 (견고성)": have[m]["res"]["a_star"].median(), "B": cur["b"].nunique()})
t4b = pd.DataFrame(rows).round(4)
t4b.to_csv(TAB_DIR / f"t4b_astar_mean_curve{tag}.csv", index=False, encoding="utf-8-sig")
print(t4b.to_string(index=False))

### 2-2. 모델 선택: 짝지은 차이 (6단계) → `selection.json`
모형마다 자기 a*(2-1)에 고정한 회차별 SR을 구하고, 같은 회차끼리 빼서 차이 B개를 만듭니다.
- ②−①의 2.5% 분위수 > 0 이면 ML 이득
- ②−③의 97.5% 분위수 < δ 이면 ③ 선택, 아니면 ② (`ALPHA` = 0.025)
- δ가 미정(`PENDING`)이면 그림 4와 같은 이유로 ②−③은 계산하지 않고 선택을 "보류"로 남깁니다 (판정 기준이 결과에 끌려가지 않게)
- OOB 구간은 정식 신뢰구간이 아니라 선택 근거입니다.

결과: `report/tables/t4c_paired_selection.csv`, `outputs/collect/selection.json` (05_test·99_freeze가 읽음)

In [ ]:
# 2-2. 짝지은 차이와 selection.json
import datetime
SEL_DIR = OUT_DIR / "collect"; SEL_DIR.mkdir(parents=True, exist_ok=True)

def sr_at_astar(m):
    """모형 m의 회차별 SR을 자기 a*에서 꺼낸다. 반환: Series(index=b)"""
    cur = have[m]["curves"]
    s = cur[np.isclose(cur["rate"], A_STAR[m])].set_index("b")["sr"]
    assert s.index.is_unique and len(s) == have[m]["curves"]["b"].nunique(), f"{m}: a*에서 회차별 SR을 못 찾음"
    return s

def paired(a, b):
    """같은 회차끼리 a − b. 반환: (차이 Series, 요약 dict)"""
    d = (sr_at_astar(a) - sr_at_astar(b)).dropna()
    q = d.quantile([ALPHA, 0.5, 1 - ALPHA])
    return d, {"n": int(len(d)), "mean": float(d.mean()), "q_lo": float(q.iloc[0]), "median": float(q.iloc[1]),
               "q_hi": float(q.iloc[2]), "negative_rounds": int((d < 0).sum())}

DELTA_PENDING = ("DELTA" in PENDING) or (DELTA is None)
comp, rows = {}, []
if "xgb" in have and "logit" in have:
    _, comp["xgb_minus_logit"] = paired("xgb", "logit")
if "xgb" in have and "xgb_mono" in have and not DELTA_PENDING:
    _, comp["xgb_minus_mono"] = paired("xgb", "xgb_mono")
for k, label in (("xgb_minus_logit", "② − ① (ML 이득)"), ("xgb_minus_mono", "② − ③ (단조 제약 비용)")):
    if k in comp:
        c = comp[k]
        rows.append({"비교": label, "회차": c["n"], "평균": c["mean"], f"{ALPHA:.1%} 분위수": c["q_lo"],
                     "중앙값": c["median"], f"{1 - ALPHA:.1%} 분위수": c["q_hi"], "음수 회차": c["negative_rounds"]})
t4c = pd.DataFrame(rows).round(4)
t4c.to_csv(TAB_DIR / f"t4c_paired_selection{tag}.csv", index=False, encoding="utf-8-sig")
print(t4c.to_string(index=False) if len(t4c) else "짝지은 비교를 할 모형이 아직 없음")

ml_gain = comp["xgb_minus_logit"]["q_lo"] > 0 if "xgb_minus_logit" in comp else None
if DELTA_PENDING:
    selected, reason = None, "보류: δ 미정 (00_config DELTA 확정 뒤 다시 실행)"
elif "xgb_minus_mono" not in comp:
    selected, reason = None, "보류: ② 또는 ③ 결과 없음"
elif comp["xgb_minus_mono"]["q_hi"] < DELTA:
    selected, reason = "xgb_mono", f"②−③ {1 - ALPHA:.1%} 분위수 {comp['xgb_minus_mono']['q_hi']:.4f} < δ {DELTA} → ③ 선택"
else:
    selected, reason = "xgb", f"②−③ {1 - ALPHA:.1%} 분위수 {comp['xgb_minus_mono']['q_hi']:.4f} ≥ δ {DELTA} → ② 선택"
if not SPLIT_OK:
    reason += " | 주의: 분할 지문 불일치·누락 (시험용)"

selection = dict(
    selected_model=selected, reason=reason, ml_gain=ml_gain, status="시험용" if FAILED else "정식",
    rule={"a_star": "평균 곡선 최대 (MIN_RATE 이상)", "ml_gain": f"②−① {ALPHA:.1%} 분위수 > 0",
          "select_mono": f"②−③ {1 - ALPHA:.1%} 분위수 < δ"},
    a_star={m: A_STAR[m] for m in A_STAR}, comparisons=comp, delta=None if DELTA_PENDING else DELTA, alpha=ALPHA,
    min_rate=MIN_RATE, B={m: int(have[m]["res"].shape[0]) for m in have},
    split_hash=split, split_ok=SPLIT_OK, config_hash=NOW["config_hash"], lib_prep_hash=NOW["lib_prep_hash"],
    created=datetime.datetime.now().isoformat(timespec="seconds"))
with open(SEL_DIR / "selection.json", "w", encoding="utf-8") as f:
    json.dump(selection, f, ensure_ascii=False, indent=1, default=str)
print("\n선택:", selected, "|", reason)
print("저장:", (SEL_DIR / "selection.json").relative_to(ROOT))

In [ ]:
# 3. 그림 2 (승인율별 SR 평균 ±1SD)와 그림 3 (회차별 a*_b 분포)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for i, (m, label) in enumerate(MODELS.items()):
    if m not in have: continue
    g = have[m]["curves"].groupby("rate")["sr"]; mu, sd = g.mean(), g.std(ddof=1).fillna(0)
    ax.plot(mu.index, mu.values, label=label, color=f"C{i}")
    ax.fill_between(mu.index, mu - sd, mu + sd, alpha=0.15, color=f"C{i}")
    ax.axvline(have[m]["res"]["a_star"].median(), ls="--", lw=0.8, color=f"C{i}")
ax.set_xlabel("승인율 a"); ax.set_ylabel("OOB SR (거절=0)"); ax.legend(frameon=False, fontsize=8)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fig2_oob_sr_curves{tag}.png", dpi=300)

fig, ax = plt.subplots(figsize=(6.5, 2.8))
for i, (m, label) in enumerate(MODELS.items()):
    if m not in have: continue
    ax.hist(have[m]["res"]["a_star"], bins=np.arange(0.30, 0.71, 0.01), histtype="step", lw=1.6, label=label, color=f"C{i}")
ax.set_xlabel("회차별 최적 승인율 a*_b"); ax.set_ylabel("회차 수"); ax.legend(frameon=False, fontsize=8)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fig3_astar_dist{tag}.png", dpi=300)
plt.show()

In [ ]:
# 4. 그림 4: 회차별 ΔSR (②의 a*_b 지점 기준, ② − 다른 모형). ②가 없으면 건너뜀
if "xgb" not in have:
    print("② 결과가 없어 그림 4는 건너뜀")
else:
    ref = have["xgb"]["curves"]; a_ref = have["xgb"]["res"].set_index("b")["a_star"]
    fig, ax = plt.subplots(figsize=(6.5, 3.2)); rows4 = []
    for i, m, label in ((2, "xgb_mono", "② − ③ (단조 제약 비용)"), (0, "logit", "② − ① (ML 이득)")):
        if m not in have: continue
        if m == "xgb_mono" and "DELTA" in PENDING:   # [미정] δ 확정 전에는 ②−③을 보지 않는다(판정 기준이 결과에 끌려가지 않게)
            print("δ 미정: ②−③ 곡선은 δ를 00_config에 확정한 뒤 다시 실행하면 그립니다"); continue
        mm = ref.merge(have[m]["curves"], on=["b", "rate"], suffixes=("_r", "_n"))
        mm["d"] = mm["sr_r"] - mm["sr_n"]
        mean = mm.groupby("rate")["d"].mean()
        ax.plot(mean.index, mean.values, label=label, color=f"C{i}")
        at = mm[np.isclose(mm["rate"], mm["b"].map(a_ref))]
        rows4.append({"비교": label, "ΔSR 중앙값(②의 a*_b)": at["d"].median(),
                      "최소": at["d"].min(), "최대": at["d"].max(), "음수 회차": int((at["d"] < 0).sum())})
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("승인율 a"); ax.set_ylabel("ΔSR (OOB, 회차 평균)")
    ax.legend(frameon=False, fontsize=8)
    for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
    fig.tight_layout(); fig.savefig(FIG_DIR / f"fig4_oob_dsr{tag}.png", dpi=300); plt.show()
    print(pd.DataFrame(rows4).round(4).to_string(index=False))

In [ ]:
# 5. 부록 C: ② 최종 모형의 PDP (부호 ±1 변수 20개, Dev 무작위 5,000건, 원 단위 5~95% 분위수 격자)
f = OUT_DIR / "xgb" / "final_model.joblib"
if not f.exists():
    print("② 최종 모형 없음 → 부록 C 건너뜀")
else:
    prep, model = joblib.load(f)
    Xs = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet").sample(5_000, random_state=SEED)
    cols = list(SIGN_OF); nr = int(np.ceil(len(cols) / 5))
    fig, axs = plt.subplots(nr, 5, figsize=(6.5, 1.5 * nr), squeeze=False); rows = []
    for ax, c in zip(axs.flat, cols):
        s_ = pdp_1d(prep, model, Xs, c)
        rho = pd.Series(s_.values).corr(pd.Series(s_.index), method="spearman") if len(s_) > 2 else np.nan
        ax.plot(s_.index, s_.values, color="C1" if SIGN_OF[c] > 0 else "C0", lw=1.2)
        ax.set_title(f"{c} ({SIGN_OF[c]:+d})", fontsize=5.5); ax.tick_params(labelsize=4.5); tidy(ax)
        rows.append(dict(변수=c, 부호=SIGN_OF[c], pdp_rho=rho, 변화폭=s_.iloc[-1] - s_.iloc[0]))
    for ax in list(axs.flat)[len(cols):]: ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / f"figA2_pdp{tag}.png", dpi=300); plt.show()
    pdt = pd.DataFrame(rows)
    pdt["판정"] = np.where(pdt["변화폭"].abs() < 1e-3, "평평(|변화폭|<0.001)", np.where(np.sign(pdt["변화폭"]) == pdt["부호"], "일치", "반대"))   # [미정·기본값] 평평 기준 0.001
    pdt.round(4).to_csv(TAB_DIR / f"tA3_pdp_check{tag}.csv", index=False, encoding="utf-8-sig")
    print(pdt.round(4).to_string(index=False))